# Matplotlib Quiz Practice

Same techniques as the lecture notebook, new data, so you can't just recall
the answer -- you have to apply the method. Series used:

| Series | What it is | Frequency |
|---|---|---|
| `GDP` | US nominal GDP, billions $ | quarterly, 1947- |
| `CPIAUCSL` | CPI, all urban consumers (price level) | monthly, 1947- |
| `US_state_GDP.xls` | State-level GDP and growth, 2009-2012 | annual, cross-section |
| `UNRATE`, `USREC` | from the lecture | monthly |

Each exercise names the lecture section it drills. Write your own code in the
empty cell below each prompt -- don't just re-run the lecture's cells. A
separate `..._SOLUTIONS.ipynb` has worked answers; don't open it until
you've given each exercise a real attempt.

Rule of thumb for the quiz: if a prompt doesn't tell you `plt` vs object
style, use the **object style** (`fig, ax = plt.subplots()`) -- that's what
the course uses from Section 1 on.


## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

%matplotlib inline


## Part A -- Data prep in pandas (lecture Section 0)

### A1. Load and index

Load `GDP.csv` and `CPIAUCSL.csv` the same way the lecture loaded UNRATE:
`parse_dates` on the date column, that column as `index_col`, then pull out
the one data column as a Series. Name the results `gdp` and `cpi`.

Print `gdp.tail()` and note the frequency (how many months apart are the
index values?).

In [ ]:
gdp = pd.read_csv('GDP.csv', parse_dates=['observation_date'],
                   index_col='observation_date')['GDP']

cpi = pd.read_csv('CPIAUCSL.csv', parse_dates=['observation_date'],
                   index_col='observation_date')['CPIAUCSL']

print(gdp.tail())
print(cpi.tail())
# gdp is quarterly (Jan/Apr/Jul/Oct); cpi is monthly.


### A2. Year-over-year growth, two different frequencies

`GDP` is quarterly, so "a year ago" is 4 rows back; `CPIAUCSL` is monthly,
so "a year ago" is 12 rows back (same trick as `unrate.shift(12)` in the
lecture, different shift amount).

Build:
- `gdp_growth = 100 * (gdp / gdp.shift(4) - 1)`  (percent growth over 4 quarters)
- `inflation = 100 * (cpi / cpi.shift(12) - 1)`  (this *is* the standard
  inflation-rate definition, not the lecture's simple point difference)

Then build one DataFrame called `macro` with columns `GDP_GROWTH` and
`INFLATION`, aligned on their common dates (`pd.DataFrame` will outer-join
the two Series on the index automatically). Slice `macro` to start in 1960
and check `.isnull().sum()`.

In [ ]:
gdp_growth = 100 * (gdp / gdp.shift(4) - 1)
inflation = 100 * (cpi / cpi.shift(12) - 1)

macro = pd.DataFrame({'GDP_GROWTH': gdp_growth, 'INFLATION': inflation})
macro = macro.loc['1960':]
print(macro.isnull().sum())
macro.tail()


### A3. `groupby` on a cross-section

Load `US_state_GDP.xls` with `pd.read_excel` (sheet `'Sheet1'`). It has one
row per state, with columns including `state`, `region`, `gdp_growth_2012`.

Use `groupby('region')['gdp_growth_2012'].mean()` to get average 2012 GDP
growth by Census region, and store it as `region_growth`. Sort it from
highest to lowest.

In [ ]:
states = pd.read_excel('US_state_GDP.xls', sheet_name='Sheet1')

region_growth = states.groupby('region')['gdp_growth_2012'].mean()
region_growth = region_growth.sort_values(ascending=False)
region_growth


### A4. Boolean mask

From `states`, build two sub-DataFrames: `fast` (states where
`gdp_growth_2012` is above the national average `states['gdp_growth_2012'].mean()`)
and `slow` (at or below it). Print how many states are in each group.

In [ ]:
avg = states['gdp_growth_2012'].mean()
fast = states[states['gdp_growth_2012'] > avg]
slow = states[states['gdp_growth_2012'] <= avg]
print(len(fast), 'states above average,', len(slow), 'at or below')


## Part B -- Line plots (lecture Section 2)

### B1. Basic line plot, object style

Plot `gdp` (the level, not growth) as a line, object style
(`fig, ax = plt.subplots()`). Label the y-axis "Billions of dollars".

In [ ]:
fig, ax = plt.subplots()
ax.plot(gdp)
ax.set_ylabel('Billions of dollars')
plt.show()


### B2. Format string, zoomed in

Plot `inflation` from `2021` onward only, using a **red dash-dot line**
(check the format-string table: color `r`, line `-.`). This is the
2021-2022 inflation spike.

In [ ]:
recent_inf = inflation.loc['2021':]

fig, ax = plt.subplots()
ax.plot(recent_inf, 'r-.')
plt.show()


## Part C -- Scatter plots (lecture Section 3)

### C1. Plain scatter with a reference line

Scatter `macro['GDP_GROWTH']` (x) against `macro['INFLATION']` (y). Use
`s=10, alpha=.4` like the lecture. Add a horizontal reference line at 0 with
`ax.axhline`, and label both axes.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(macro['GDP_GROWTH'], macro['INFLATION'], s=10, alpha=.4)
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_xlabel('GDP growth, y/y (%)')
ax.set_ylabel('Inflation, y/y (%)')
plt.show()


### C2. Color-mapped scatter with a colorbar

Same scatter as C1, but color each point by `macro.index.year` with
`cmap='viridis'`, and add a colorbar labeled "Year" (same pattern as the
lecture's `fig.colorbar(points, ax=ax, label=...)`).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
points = ax.scatter(macro['GDP_GROWTH'], macro['INFLATION'],
                     c=macro.index.year, s=10, cmap='viridis')
fig.colorbar(points, ax=ax, label='Year')
ax.set_xlabel('GDP growth, y/y (%)')
ax.set_ylabel('Inflation, y/y (%)')
plt.show()


### C3. Split scatter with a legend

Using the `states` DataFrame, scatter `gdp_growth_2009` (x) against
`gdp_growth_2012` (y), but split the points into two `ax.scatter` calls by
region: Northeast (`region == 'NE'`) in one color with `label='Northeast'`,
everywhere else in another color with `label='Other region'`. Add
`ax.legend()`.

In [ ]:
ne = states[states['region'] == 'NE']
other = states[states['region'] != 'NE']

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(other['gdp_growth_2009'], other['gdp_growth_2012'], s=25, alpha=0.6,
           color='#9AAAB1', label='Other region')
ax.scatter(ne['gdp_growth_2009'], ne['gdp_growth_2012'], s=25, alpha=0.9,
           color='#C4602A', label='Northeast')
ax.set_xlabel('GDP growth 2009 (%)')
ax.set_ylabel('GDP growth 2012 (%)')
ax.legend()
plt.show()


## Part D -- Bar charts, histograms, pie (lecture Section 4)

### D1. Vertical bar chart

Bar chart of `region_growth` (from A3): one bar per region, average 2012 GDP
growth. Label the y-axis.

In [ ]:
fig, ax = plt.subplots()
ax.bar(region_growth.index, region_growth.values, color='#2F6DB5')
ax.set_ylabel('Average GDP growth, 2012 (%)')
plt.show()


### D2. Horizontal bar, sorted, highest on top

Same data as D1, but `ax.barh`, and use `ax.invert_yaxis()` so the highest
region is at the top (`region_growth` is already sorted descending from
A3 -- if yours isn't, sort it first).

In [ ]:
fig, ax = plt.subplots()
ax.barh(region_growth.index, region_growth.values, color='#2F6DB5')
ax.invert_yaxis()
ax.set_xlabel('Average GDP growth, 2012 (%)')
plt.show()


### D3. Histogram

Histogram of `inflation.dropna()` with `bins=50`. Label the x-axis "Year
over year inflation (%)".

In [ ]:
fig, ax = plt.subplots()
ax.hist(inflation.dropna(), bins=50)
ax.set_xlabel('Year over year inflation (%)')
ax.set_ylabel('Number of months')
plt.show()


### D4. Histogram + normal curve overlay

Same idea as the lecture's normal-curve overlay, but for
`states['gdp_growth_2012']` (51 observations, so use a coarser bin count,
e.g. `bins=12`, and `density=True`). Compute the mean and sd of the column,
build a `grid = np.linspace(...)` spanning a bit past the data's min and
max, and plot the normal density on top in a contrasting color with a
legend.

In [ ]:
g2012 = states['gdp_growth_2012']
m, sd = g2012.mean(), g2012.std()

grid = np.linspace(g2012.min() - 1, g2012.max() + 1, 200)
normal = np.exp(-0.5 * ((grid - m) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))

fig, ax = plt.subplots()
ax.hist(g2012, bins=12, density=True, color='#9AAAB1', label='Data')
ax.plot(grid, normal, color='#C4602A', linewidth=2,
        label='Normal, same mean and sd')
ax.set_xlabel('State GDP growth, 2012 (%)')
ax.legend()
plt.show()


### D5. Pie chart

Pie chart of the `fast` / `slow` counts from A4: share of states above vs.
at-or-below the national average 2012 growth rate. Use `autopct='%1.0f%%'`
like the lecture.

In [ ]:
fig, ax = plt.subplots()
ax.pie([len(fast), len(slow)], labels=['Above average', 'At or below average'],
       autopct='%1.0f%%', colors=['#C4602A', '#9AAAB1'])
plt.show()


## Part E -- Several lines, log scale (lecture Section 5)

### E1. Two lines, two different units (spot the problem)

Plot `gdp` and `cpi` on the *same* axes, each with a `label`, and add a
legend. Don't fix anything yet -- just look at it.

**Question:** `GDP` is in billions of dollars and `CPIAUCSL` is an index
around 20-330. What does this chart get wrong, and what would you do about
it (two options: two y-axes with `ax.twinx()`, or subplots)?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(gdp, label='GDP ($bn)')
ax.plot(cpi, label='CPI (index)')
ax.legend()
plt.show()

# Problem: CPI is invisible -- it's squashed near zero because GDP's scale
# (hundreds to tens of thousands) dwarfs CPI's scale (tens to low hundreds).
# Fix: ax.twinx() for a second y-axis, or two separate subplots.


### E2. Log scale

Plot `gdp` on a **linear** y-axis (left panel) and a **log** y-axis (right
panel), side by side with `plt.subplots(1, 2, ...)`, same pattern as the
lecture's compound-growth example (`axes[1].set_yscale('log')`).

**Question:** Two recessions are visible as dips. Which one looks
proportionally bigger on the log panel than it does on the linear panel,
and why does log scale change the answer?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(gdp)
axes[0].set_title('Linear scale')

axes[1].plot(gdp)
axes[1].set_yscale('log')
axes[1].set_title('Log scale')

plt.show()

# 2020 (COVID) is a short, sharp drop from a much higher base than, say,
# 2008-09, so on a *linear* axis it doesn't look dramatic -- but on a log
# axis, which shows percentage change as equal vertical distance, the 2020
# drop reads as a much steeper fall, because it happened faster even if not
# deeper in percentage terms than 2008-09.


## Part F -- Subplots (lecture Section 6)

### F1. 2x2 subplots, written out by hand

Make a 2x2 grid of panels (`plt.subplots(2, 2, figsize=(11, 7), sharey=True)`),
each showing `inflation` over one high-inflation window:

- `axes[0, 0]`: `'1973':'1975'`, title `'1973-75 oil shock'`
- `axes[0, 1]`: `'1979':'1982'`, title `'Volcker disinflation runs up first'`
- `axes[1, 0]`: `'2007':'2009'`, title `'2008-09'`
- `axes[1, 1]`: `'2021':'2023'`, title `'2021-22'`

Finish with `fig.tight_layout()`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=True)

axes[0, 0].plot(inflation.loc['1973':'1975'])
axes[0, 0].set_title('1973-75 oil shock')

axes[0, 1].plot(inflation.loc['1979':'1982'])
axes[0, 1].set_title('Volcker disinflation runs up first')

axes[1, 0].plot(inflation.loc['2007':'2009'])
axes[1, 0].set_title('2008-09')

axes[1, 1].plot(inflation.loc['2021':'2023'])
axes[1, 1].set_title('2021-22')

fig.tight_layout()
plt.show()


### F2. The same chart, built with a loop

Rebuild F1 using the lecture's loop pattern over `axes.flat`: one list of
start years, one of end years, one of titles, and a `for i in range(4)`
loop that plots `inflation.loc[starts[i]:ends[i]]` into `axes.flat[i]` and
sets its title. Set the y-label on `axes[0, 0]` and `axes[1, 0]` only.

In [ ]:
starts = ['1973', '1979', '2007', '2021']
ends = ['1975', '1982', '2009', '2023']
titles = ['1973-75 oil shock', 'Volcker disinflation runs up first',
          '2008-09', '2021-22']

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=True)

for i in range(len(starts)):
    window = inflation.loc[starts[i]:ends[i]]
    axes.flat[i].plot(window, color='#2F6DB5', linewidth=1.2)
    axes.flat[i].set_title(titles[i])

axes[0, 0].set_ylabel('Inflation, y/y (%)')
axes[1, 0].set_ylabel('Inflation, y/y (%)')
fig.tight_layout()
plt.show()


## Part G -- Dates, shaded recessions, annotation (lecture Section 7-8)

### G1. Shade recessions behind a quarterly series

`USREC` is monthly and `gdp_growth` is quarterly, so they don't share an
index yet. Resample `usrec` to quarterly with
`usrec.resample('QS').max()` (1 if *any* month in the quarter was a
recession month), call it `usrec_q`. Then build
`gdp_plot = pd.DataFrame({'GDP_GROWTH': gdp_growth, 'USREC': usrec_q}).dropna()`.

Plot `gdp_plot['GDP_GROWTH']` as a black line, and use `ax.fill_between`
with `where=gdp_plot['USREC'] == 1` to shade recession quarters in grey
(same pattern as the lecture's unemployment chart -- pick sensible
`y1`/`y2` bounds from the data's min/max, and match them with
`ax.set_ylim`).

In [ ]:
usrec = pd.read_csv('USREC.csv', parse_dates=['observation_date'],
                     index_col='observation_date')['USREC']
usrec_q = usrec.resample('QS').max()

gdp_plot = pd.DataFrame({'GDP_GROWTH': gdp_growth, 'USREC': usrec_q}).dropna()

ymin, ymax = gdp_plot['GDP_GROWTH'].min() - 1, gdp_plot['GDP_GROWTH'].max() + 1

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(gdp_plot.index, gdp_plot['GDP_GROWTH'], color='black', linewidth=1)
ax.fill_between(gdp_plot.index, ymin, ymax, where=gdp_plot['USREC'] == 1,
                color='grey', alpha=.3, label='NBER recession')
ax.set_ylim(ymin, ymax)
ax.set_xlim(gdp_plot.index[0], gdp_plot.index[-1])
ax.set_ylabel('GDP growth, y/y (%)')
ax.legend(loc='lower left')
plt.show()


### G2. Locator and formatter, zoomed in

Plot `inflation.loc['2021-01':'2023-12']` with markers
(`marker='o', markersize=3`). Put a tick every 3 months
(`mdates.MonthLocator(interval=3)`), format it as `'%b %Y'`, and call
`fig.autofmt_xdate()` to rotate the labels.

In [ ]:
window = inflation.loc['2021-01':'2023-12']

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(window, marker='o', markersize=3)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
fig.autofmt_xdate()
ax.set_ylabel('Inflation, y/y (%)')
plt.show()


### G3. One finished, explanatory chart

Build one chart that makes a single claim about inflation, the way the
lecture's Section 8 chart made a claim about the speed of the two
recoveries. Suggested claim: **"Inflation peaked in 2022 at roughly 3x the
Fed's 2% target, the highest reading since the early 1980s."**

Requirements (see the lecture's "what changed and why" table in Section 8):
- `fig.suptitle(...)` states the finding as a sentence, not a variable name
- `ax.set_title(..., loc='left', fontsize=9, color='dimgray')` as a subtitle
  giving units/frequency/source
- One `ax.annotate(...)` pointing at the 2022 peak with the peak value and
  date
- A horizontal reference line at `y=2` for the Fed's target, labeled in the
  legend
- Top and right spines removed
- A small `fig.text(...)` source note at the bottom

You'll need the peak value and its date first -- get those with
`inflation.idxmax()` and `inflation.max()` (restrict to `.loc['1980':]` if
an earlier reading is higher and you want the 2022 one specifically).

In [ ]:
peak_date = inflation.loc['2020':].idxmax()
peak_val = inflation.loc['2020':].max()

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(inflation.index, inflation, color='#2F6DB5', linewidth=1.3,
        label='Inflation, y/y')
ax.axhline(2, color='#9AAAB1', linewidth=1.2, linestyle='--',
           label="Fed's 2% target")

ax.scatter([peak_date], [peak_val], color='#C4602A', zorder=3, s=30)
ax.annotate(f'{peak_val:.1f}% in {peak_date.strftime("%b %Y")}',
            xy=(peak_date, peak_val), xytext=(-60, 25),
            textcoords='offset points', ha='left', fontsize=9,
            arrowprops={'arrowstyle': '->', 'color': '#C4602A'})

fig.suptitle("Inflation peaked in 2022 at roughly 3x the Fed's 2% target",
             x=0.06, ha='left', fontsize=14, fontweight='bold')
ax.set_title('US CPI inflation, year over year, percent, monthly. '
             "Dashed line: the Fed's 2% target.",
             loc='left', fontsize=9, color='dimgray')

ax.set_ylabel('Percent')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper left', frameon=False)

fig.text(0.06, 0.01, 'Source: BLS via FRED (CPIAUCSL).', fontsize=8,
         color='dimgray')
fig.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()


## Part H -- House style and saving (lecture Section 9)

### H1. A house-style function

Write a function `econ_axes(figsize=(9, 4.5))` that returns `fig, ax` from
`plt.subplots(figsize=figsize)`, with the top and right spines removed and
a light y-axis grid (`ax.grid(axis='y', alpha=0.3)`) -- same idea as the
lecture's version. Use it to plot `gdp_growth`.

In [ ]:
def econ_axes(figsize=(9, 4.5)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    return fig, ax

fig, ax = econ_axes()
ax.plot(gdp_growth, color='#2F6DB5')
ax.set_ylabel('GDP growth, y/y (%)')
plt.show()


### H2. Save a figure

Re-plot the chart from H1 (or reuse `fig`) and save it as both
`gdp_growth.png` (with `dpi=200, bbox_inches='tight'`) and
`gdp_growth.pdf`. Close the figure after saving.

In [ ]:
fig, ax = econ_axes()
ax.plot(gdp_growth, color='#2F6DB5')
ax.set_ylabel('GDP growth, y/y (%)')

fig.savefig('gdp_growth.png', dpi=200, bbox_inches='tight')
fig.savefig('gdp_growth.pdf', bbox_inches='tight')
plt.close(fig)
print('saved gdp_growth.png and gdp_growth.pdf')
